### Import Libraries

In [24]:
from roboflow import Roboflow
from ultralytics import YOLO
import cv2

### Download ID Detection Dataset

In [ ]:
rf = Roboflow(api_key="1z1P2cHwRlZIsTCfHZ3m")
project = rf.workspace("tutorocean").project("egyptianid-detection")
version = project.version(1)
dataset = version.download("yolov8")

### Fine Tune YOLOv8n

In [ ]:
model = YOLO('yolov8n.pt').load('runs/detect/train2/weights/best.pt')

model.train(data='EgyptianID-Detection-1/data.yaml', epochs=30, imgsz=640, amp=False)

### Save The Model

In [3]:
model.save('ID_detection.pt')

### Download Line Detection on Card Dataset

In [ ]:
rf = Roboflow(api_key="1z1P2cHwRlZIsTCfHZ3m")
project = rf.workspace("elsoudy").project("card-hleyg")
version = project.version(1)
dataset = version.download("yolov8")

### Fine Tune YOLOv8n

In [ ]:
model_l = YOLO('yolov8n.pt')

model_l.train(data='card-1/data.yaml', epochs=40, imgsz=640, amp=False)

### Save the Model

In [4]:
model_l.save('line_detection.pt')

### Predict on Image

In [ ]:
id_card_model = YOLO('ID_detection.pt')
line_detection_model = YOLO('line_detection.pt')

# Load the image
image = cv2.imread('EgyptianID-Detection-1/test/images/7_jpg.rf.0abd228e6bebb89a510e50a6504d0726.jpg')
image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
id_results = id_card_model(image)

# Extract the ID card bounding box
for box in id_results[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    cropped_id_card = image[y1:y2, x1:x2]

    # Step 2: Detect text lines within the ID card
    line_results = line_detection_model(cropped_id_card)

    # Crop and save each detected line
    for i, line_box in enumerate(line_results[0].boxes):
        lx1, ly1, lx2, ly2 = map(int, line_box.xyxy[0])
        cropped_line = cropped_id_card[ly1:ly2, lx1:lx2]
        cv2.imwrite(f'line_{i}.png', cropped_line)